In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

base_path = Path(r"C:\FeedbackControl\pyclm\data\mdck\barspeeds")

track_path = base_path / "tracks"
save_path = base_path / "figures"

import matplotlib as mpl

mpl.rc("axes", facecolor="#ffffff00", grid=False, edgecolor="k", labelcolor="k")
mpl.rc("figure", facecolor="#00000000", dpi=100)
mpl.rc(
    "axes.spines",
    top=False,
    right=False,
)
mpl.rc("xtick", color="k", bottom=True)
mpl.rc("ytick", color="k", left=True)

In [ ]:
tracks = {}
stems = []
files = []

for f in track_path.glob("*.csv"):
    files.append(f)
    stems.append(f.stem)
    tracks[f.stem] = pd.read_csv(f)

In [ ]:
px_um = 1.33
frame_min = 5
img_shape = (800, 800)

for df in tracks.values():
    df["px_x_centered"] = df["px_x"] - img_shape[1] / 2
    df["px_y_centered"] = df["px_y"] - img_shape[0] / 2

    df["um_x"] = df["px_x"] * px_um
    df["um_y"] = df["px_y"] * px_um

    df["um_x_centered"] = df["px_x_centered"] * px_um
    df["um_y_centered"] = df["px_y_centered"] * px_um

    df["time"] = df["frame"] * frame_min

In [ ]:
import tifffile
from scipy.spatial.distance import cdist

k = 13


def plot_vertical_velocities(k, time, ax, image=True, pattern=True, vectors=True):
    n_samples = 55
    time_limit = 7.5
    sigma = 20

    print(stems[k])
    df = tracks[stems[k]]

    df["dx"] = df.groupby("track_id")["um_x"].diff() / frame_min
    df["dy"] = df.groupby("track_id")["um_y"].diff() / frame_min
    df["dtot"] = np.sqrt(df["dx"] ** 2 + df["dy"] ** 2)

    # get data points near time, filter out bad tracks
    df = df[(df["time"] - time).abs() < time_limit]
    df = df[~df["dx"].isna()]
    df = df[~df["dy"].isna()]
    df = df[df["dtot"] < 2].copy()

    # get sampling positions in image coordinates and um coordinates
    xx = np.linspace(-img_shape[1] / 2, img_shape[1] / 2, n_samples) * px_um
    yy = np.linspace(-img_shape[0] / 2, img_shape[0] / 2, n_samples) * px_um

    px_x = np.linspace(0, img_shape[1], n_samples)
    px_y = np.linspace(0, img_shape[0], n_samples)

    xx, yy = np.meshgrid(xx, yy)
    X = np.array([xx.flatten(), yy.flatten()]).T

    px_xx, px_yy = np.meshgrid(px_x, px_y)
    X_px = np.array([px_xx.flatten(), px_yy.flatten()])

    # get dxdy at sample positions using gaussian kernels
    pos = df[["um_x_centered", "um_y_centered"]].values
    distances = cdist(X, pos)
    weights = np.exp(-0.5 * np.square(distances) / np.square(sigma))

    dx_weighted = np.mean(weights * df["dx"].values[None, :], axis=1) / np.mean(
        weights, axis=1
    )
    dy_weighted = np.mean(weights * df["dy"].values[None, :], axis=1) / np.mean(
        weights, axis=1
    )
    area_weighted = np.mean(weights * df["area"].values[None, :], axis=1) / np.mean(
        weights, axis=1
    )

    # show light bars
    frame_data, frame_pattern = tifffile.imread(base_path / f"{stems[k][:-7]}.tif")[
        int(time // frame_min)
    ]
    frame_data = 1 - np.clip((frame_data - 1000) / 7000, 0, 1)
    frame_pattern = frame_pattern / 255

    vmin, vmax = -30, 30

    cmap = sns.diverging_palette(28, 10, 94, 55, sep=1, center="light", as_cmap=True)
    ax3 = plt.imshow(
        dy_weighted.reshape(n_samples, n_samples) * 60,
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        extent=(0, img_shape[0], 0, img_shape[1]),
        origin="lower",
    )

    if image:
        ax.imshow(np.stack([frame_data, frame_data, frame_data], axis=-1))
    if pattern:
        ax.imshow(
            np.stack(
                [
                    frame_pattern * 39 / 255,
                    frame_pattern * 170 / 255,
                    frame_pattern * 225 / 255,
                    frame_pattern * 40 / 100,
                ],
                axis=-1,
            ),
            origin="lower",
        )

    if not (image or pattern):
        ax.imshow(
            np.stack(
                [
                    np.zeros_like(frame_pattern),
                    np.zeros_like(frame_pattern),
                    np.zeros_like(frame_pattern),
                    np.zeros_like(frame_pattern),
                ],
                axis=-1,
            )
        )

    if vectors:
        q = ax.quiver(
            X_px[0],
            X_px[1],
            dx_weighted * 30,
            dy_weighted * 30,
            color="k",
            scale_units="xy",
            clim=(vmin, vmax),
            angles="xy",
            scale=1,
            lw=2,
            alpha=1.0,
        )

        fig2, ax2 = plt.subplots(1, 1, figsize=(0.6, 4))

        cbar = fig2.colorbar(ax3, orientation="vertical", cax=ax2)
        cbar.set_label("Local Velocity ($\mu$m/h)")

        plt.savefig(save_path / "local_velocity_colorbar.pdf", bbox_inches="tight")

    ax.set_xlim(150, 650)
    ax.set_ylim(150, 650)
    ax.set_xticks([])
    ax.set_yticks([])

    return ax


# fig, ax = plt.subplots(1, 1, figsize=(4, 3))
# plot_vertical_velocities(13, 900, ax, image=False)
#
# plt.show()

fig, ax = plt.subplots(1, 1, figsize=(4, 3))
plot_vertical_velocities(2, 300, ax, image=False)
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)
fig.savefig(save_path / f"slow_vectors_noimage.pdf")

plt.show()

fig, ax = plt.subplots(1, 1, figsize=(4, 3))
plot_vertical_velocities(2, 360, ax, image=False)
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)
fig.savefig(save_path / f"slow_vectors_noimage.pdf")

plt.show()

fig, ax = plt.subplots(1, 1, figsize=(4, 3))
plot_vertical_velocities(2, 300, ax, image=True, vectors=False)
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)
fig.savefig(save_path / f"slow_vectors_with_image.png", dpi=300)

plt.show()

fig, ax = plt.subplots(1, 1, figsize=(4, 3))
plot_vertical_velocities(14, 600, ax, image=False)
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)
fig.savefig(save_path / f"fast_vectors_noimage.pdf")

plt.show()

fig, ax = plt.subplots(1, 1, figsize=(4, 3))
plot_vertical_velocities(14, 600, ax, image=True, vectors=False)
ax.spines["top"].set_visible(True)
ax.spines["right"].set_visible(True)
fig.savefig(save_path / f"fast_vectors_with_image.png", dpi=300)

plt.show()

In [ ]:
from collections import defaultdict


def plot_over_phase(ax, k_vals, metric="vertical speed mean", color="b"):
    avg_velocities = defaultdict(list)

    for k in k_vals:
        print(stems[k])
        df = tracks[stems[k]]

        df["dx"] = df.groupby("track_id")["um_x"].diff() / frame_min
        df["dy"] = df.groupby("track_id")["um_y"].diff() / frame_min
        df["dtot"] = np.sqrt(df["dx"] ** 2 + df["dy"] ** 2)
        df["normalized_area"] = df["area"] / df.groupby("track_id")["area"].transform(
            "mean"
        )

        assert stems[k][:3] == "bar"

        bar_speed_str = stems[k].split(".")[0][3:]
        bar_speed = int(bar_speed_str) / (10 ** (len(bar_speed_str) - 1))
        period_time = 100 / bar_speed

        df["phase"] = (
            (df["time"] - (df["um_y"] / bar_speed)) % (period_time)
        ) / period_time
        df["phase"] = (-(df["phase"] - 0.1) * 2 * np.pi) % (2 * np.pi)
        df["phase"] = df["phase"] * (df["phase"] < np.pi) - (
            2 * np.pi - df["phase"]
        ) * (df["phase"] >= np.pi)

        df_cropped = df[
            df["time"].between(180, 900)
            & df["px_x"].between(150, 650)
            & df["px_y"].between(150, 650)
            & (df["dtot"] < 2)
            & (~df["dx"].isna())
            & (~df["dy"].isna())
        ].copy()

        bw = np.pi / 5
        df_cropped["phase_bin"] = ((df_cropped["phase"] - bw / 2) // bw) * bw + bw
        df_cropped["vertical_velocity"] = df_cropped["dy"] * 60

        t = df_cropped.groupby("phase_bin")[metric].mean().reset_index()
        avg_velocities["phase_bin"].extend(t["phase_bin"])
        avg_velocities[metric].extend(t[metric])
        avg_velocities["replicate"].extend([stems[k].split(".")[2][:2]] * len(t))

    sns.lineplot(
        pd.DataFrame(avg_velocities),
        x="phase_bin",
        y=metric,
        errorbar=("sd", 1),
        color=color,
        lw=3,
        ax=ax,
    )


colors = {"fast": "sandybrown", "slow": "lightcoral"}

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
plot_over_phase(ax, [12, 13, 14], metric="vertical_velocity", color=colors["fast"])

plt.axhline(0, color="k", linestyle="--")
plt.ylabel("Tissue Velocity (um/h)")
plt.xlabel("Phase")
plt.xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi], ["-π", "-π/2", "0", "π/2", "π"])

plt.savefig(save_path / f"velocity_vs_phase_60umh.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(4, 3))
plot_over_phase(ax, [0, 1, 3], metric="vertical_velocity", color=colors["slow"])

# plt.legend(title="replicate")
plt.axhline(0, color="k", linestyle="--")
plt.ylabel("Tissue Velocity (um/h)")
plt.xlabel("Phase")
plt.xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi], ["-π", "-π/2", "0", "π/2", "π"])
plt.savefig(save_path / f"velocity_vs_phase_15umh.pdf", bbox_inches="tight")
plt.show()

plt.show()
fig, ax = plt.subplots(figsize=(8, 6))
plot_over_phase(ax, [0, 1, 3], metric="vertical_velocity", color=colors["slow"])
plot_over_phase(ax, [12, 13, 14], metric="vertical_velocity", color=colors["fast"])

plt.axhline(0, color="k", linestyle="--")
plt.ylabel("Tissue Velocity (um/h)")
plt.xlabel("Phase")
plt.xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi], ["-π", "-π/2", "0", "π/2", "π"])
plt.savefig(save_path / f"velocity_vs_phase_fastandslow.pdf", bbox_inches="tight")
plt.show()

In [ ]:
plt.show()
from collections import defaultdict
from itertools import pairwise

from statannotations.Annotator import Annotator


def plot_bars(ax, k_vals, pal, metric="vertical speed mean"):
    avg_velocities = defaultdict(list)

    bar_speeds = set()

    for k in k_vals:
        print(stems[k])
        df = tracks[stems[k]]

        df["dx"] = df.groupby("track_id")["um_x"].diff() / frame_min
        df["dy"] = df.groupby("track_id")["um_y"].diff() / frame_min
        df["dtot"] = np.sqrt(df["dx"] ** 2 + df["dy"] ** 2)
        df["normalized_area"] = df["area"] / df.groupby("track_id")["area"].transform(
            "mean"
        )

        assert stems[k][:3] == "bar"
        df_cropped = df[
            df["time"].between(180, 900)
            & df["px_x"].between(150, 650)
            & df["px_y"].between(150, 650)
            & (df["dtot"] < 2)
            & (~df["dx"].isna())
            & (~df["dy"].isna())
        ].copy()

        bar_speed_str = stems[k].split(".")[0][3:]
        bar_speed = int(bar_speed_str) / (10 ** (len(bar_speed_str) - 1))

        df_cropped["vertical_velocity"] = df_cropped["dy"] * 60

        avg_velocities[metric].append(df_cropped[metric].mean())
        avg_velocities["replicate"].append(stems[k].split(".")[2][:2])
        avg_velocities["bar_speed"].append(int(bar_speed * 60))

        bar_speeds.add(int(bar_speed * 60))

    v = pd.DataFrame(avg_velocities)
    print(v.groupby("bar_speed")["vertical_velocity"].mean())
    print(v.groupby("bar_speed")["vertical_velocity"].std())

    sns.barplot(
        pd.DataFrame(avg_velocities),
        x="bar_speed",
        y=metric,
        hue="bar_speed",
        errorbar=None,
        palette=pal,
        lw=1,
        ax=ax,
        edgecolor="k",
        legend=False,
    )
    sns.stripplot(
        pd.DataFrame(avg_velocities),
        x="bar_speed",
        y=metric,
        hue="bar_speed",
        palette="dark:k",
        ax=ax,
        legend=False,
    )

    a = Annotator(
        ax,
        [(p1, p2) for p1, p2 in pairwise(bar_speeds)],
        data=v,
        x="bar_speed",
        y=metric,
    )
    a.configure(test="t-test_paired")
    a.apply_and_annotate()


colors = {"fast": "sandybrown", "slow": "lightcoral"}

fig, ax = plt.subplots(figsize=(1.8, 5))
plot_bars(
    ax,
    [0, 1, 3, 12, 13, 14],
    metric="vertical_velocity",
    pal={15: colors["slow"], 60: colors["fast"]},
)


plt.ylabel("Mean Tissue Velocity (μm/hr)")
plt.xlabel("Bar Velocity (μm/hr)")
plt.savefig(save_path / "barplot_tissue_velocity.pdf", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(4, 3))
plot_over_phase(ax, [0, 1, 3], metric="normalized_area", color=colors["slow"])
plot_over_phase(ax, [12, 13, 14], metric="normalized_area", color=colors["fast"])

# plt.axhline(0, color="k", linestyle="--")
plt.ylabel("Normalized Cell Area")
plt.xlabel("Phase")
plt.xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi], ["-π", "-π/2", "0", "π/2", "π"])
plt.axhline(1, color="k", linestyle="--")

plt.savefig(save_path / f"normarea_vs_phase_fastandslow.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(4, 3))
plot_over_phase(ax, [0, 1, 3], metric="normalized_area", color=colors["slow"])

# plt.legend(title="replicate")
plt.ylabel("Cell Area (Normalized)")
plt.xlabel("Phase")
plt.xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi], ["-π", "-π/2", "0", "π/2", "π"])
plt.axhline(1, color="k", linestyle="--")
plt.savefig(save_path / f"normarea_vs_phase_15umh.pdf", bbox_inches="tight")
plt.show()

fig, ax = plt.subplots(figsize=(4, 3))
plot_over_phase(ax, [12, 13, 14], metric="normalized_area", color=colors["fast"])

# plt.legend(title="replicate")
plt.ylabel("Cell Area (Normalized)")
plt.xlabel("Phase")
plt.xticks([-np.pi, -np.pi / 2, 0, np.pi / 2, np.pi], ["-π", "-π/2", "0", "π/2", "π"])
plt.axhline(1, color="k", linestyle="--")
plt.savefig(save_path / f"normarea_vs_phase_60umh.pdf", bbox_inches="tight")
plt.show()

In [ ]:
def plot_single_cell_tracks(ax, k, color="b", condition="dy_hr"):
    df = tracks[stems[k]]

    df["dtime"] = df.groupby("track_id")["time"].diff()
    df["dx"] = (
        df.groupby("track_id")["um_x"].diff() / df.groupby("track_id")["time"].diff()
    )
    df["dy"] = (
        df.groupby("track_id")["um_y"].diff() / df.groupby("track_id")["time"].diff()
    )
    df["dy_hr"] = df["dy"] * 60
    df["dtot"] = np.sqrt(df["dx"] ** 2 + df["dy"] ** 2)
    df["normalized_area"] = df["area"] / df.groupby("track_id")["area"].transform(
        "mean"
    )

    assert stems[k][:3] == "bar"

    bar_speed_str = stems[k].split(".")[0][3:]
    bar_speed = int(bar_speed_str) / (10 ** (len(bar_speed_str) - 1))
    period_time = 100 / bar_speed

    df["phase"] = (
        (df["time"] - (df["um_y"] / bar_speed)) % (period_time)
    ) / period_time
    df["phase"] = (-(df["phase"] - 0.1) * 2 * np.pi) % (2 * np.pi)
    df["phase"] = df["phase"] * (df["phase"] < np.pi) - (2 * np.pi - df["phase"]) * (
        df["phase"] >= np.pi
    )
    df["phase_abs"] = np.abs(df["phase"])

    df_cropped = df[
        df["time"].between(180, 900)
        & df["px_x"].between(150, 650)
        & df["px_y"].between(150, 650)
        & (df["dtot"] < 2)
        & (~df["dx"].isna())
        & (~df["dy"].isna())
    ].copy()

    df_cropped["track_start"] = df_cropped["track_id"].map(
        df_cropped.groupby("track_id")["time"].min()
    )
    df_cropped["track_end"] = df_cropped["track_id"].map(
        df_cropped.groupby("track_id")["time"].max()
    )

    df_cropped["dtime"] = df_cropped.groupby("track_id")["time"].diff()

    bw = np.pi / 5
    df_cropped["phase_bin"] = ((df_cropped["phase"] - bw / 2) // bw) * bw + bw

    df_cropped["track_time_zero"] = df_cropped.loc[
        df_cropped["track_id"]
        .map(df_cropped.groupby("track_id")["phase_abs"].idxmin())
        .values,
        "time",
    ].values

    df_cropped["rel_time"] = (df_cropped["time"] - df_cropped["track_time_zero"]) / 60
    df_cropped["cond_smooth"] = (
        df_cropped.groupby("track_id")
        .rolling(window=10, min_periods=1, center=True)[condition]
        .mean()
        .reset_index(level=0, drop=True)
    )

    t = df_cropped.groupby("track_id")

    at_least_6h = (t["time"].max() - t["time"].min()) > 360
    no_jumps = t["dtime"].max() <= 15
    starts_early = t["rel_time"].min() < -3
    ends_late = t["rel_time"].max() > 3
    not_crazy_high = t[condition].max() < df_cropped[condition].quantile(0.999)
    not_crazy_low = t[condition].min() > df_cropped[condition].quantile(0.001)

    good_track_ids = (
        t["time"]
        .mean()
        .index[
            at_least_6h
            & no_jumps
            & starts_early
            & ends_late
            & not_crazy_high
            & not_crazy_low
        ]
    )

    print(len(good_track_ids))

    # track_ids = np.random.choice(df_cropped["track_id"].unique(), 100, False)
    track_ids = good_track_ids
    for track_id in track_ids:
        track_data = df_cropped[df_cropped["track_id"] == track_id]
        ax.plot(
            track_data["rel_time"], track_data["cond_smooth"], color=color, alpha=0.2
        )

    smoothed_mean = (
        df_cropped.groupby("rel_time")[condition]
        .mean()
        .reset_index()
        .rolling(window=5, center=True)
        .mean()
        .reset_index(level=0, drop=True)
    )

    print(smoothed_mean)

    sns.lineplot(
        smoothed_mean, x="rel_time", y=condition, errorbar=None, color="k", lw=3, ax=ax
    )


fig, ax = plt.subplots(1, 1, figsize=(4, 3))
plot_single_cell_tracks(ax, 2, colors["slow"], "dy_hr")
plt.xlabel(r"Time Since $\theta=0$ (hr)")
plt.xlim(-3, 3)
plt.ylabel("Vertical Velocity (um/h)")
plt.axhline(0, color="k", linestyle="--")
plt.savefig(save_path / f"velocity_single_cell_tracks_15umh.pdf", bbox_inches="tight")
plt.savefig(
    save_path / f"velocity_single_cell_tracks_15umh.png", dpi=300, bbox_inches="tight"
)
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(4, 3))
plot_single_cell_tracks(ax, 13, colors["fast"], "dy_hr")
plt.xlabel(r"Time Since $\theta=0$ (hr)")
plt.xlim(-3, 3)
plt.ylabel("Vertical Velocity (um/h)")
plt.axhline(0, color="k", linestyle="--")
plt.savefig(save_path / f"velocity_single_cell_tracks_60umh.pdf", bbox_inches="tight")
plt.savefig(
    save_path / f"velocity_single_cell_tracks_60umh.png", dpi=300, bbox_inches="tight"
)
plt.show()


fig, ax = plt.subplots(1, 1, figsize=(4, 3))
plot_single_cell_tracks(ax, 2, colors["slow"], "normalized_area")
plt.xlabel(r"Time Since $\theta=0$ (hr)")
plt.xlim(-3, 3)
plt.ylabel("Area (normalized)")
plt.savefig(save_path / f"area_single_cell_tracks_15umh.pdf", bbox_inches="tight")
plt.savefig(
    save_path / f"area_single_cell_tracks_15umh.png", dpi=300, bbox_inches="tight"
)
plt.show()

fig, ax = plt.subplots(1, 1, figsize=(4, 3))
plt.tight_layout(h_pad=1.5, w_pad=1.5)
plot_single_cell_tracks(ax, 13, colors["fast"], "normalized_area")
plt.xlabel(r"Time Since $\theta=0$ (hr)")
plt.xlim(-3, 3)
plt.ylabel("Area (normalized)")
plt.savefig(save_path / f"area_single_cell_tracks_60umh.pdf", bbox_inches="tight")
plt.savefig(
    save_path / f"area_single_cell_tracks_60umh.png", dpi=300, bbox_inches="tight"
)
plt.show()

In [ ]:
from collections import defaultdict

data = defaultdict(list)

for k in range(len(stems)):
    n_samples = 75
    time = 900
    time_limit = 7.5
    sigma = 10

    print(stems[k])
    df = tracks[stems[k]]

    df["dx"] = df.groupby("track_id")["um_x"].diff() / frame_min
    df["dy"] = df.groupby("track_id")["um_y"].diff() / frame_min
    df["dtot"] = np.sqrt(df["dx"] ** 2 + df["dy"] ** 2)

    if stems[k].split(".")[0] == "dark":
        bar_speed = 0
        df["phase"] = 0
    elif stems[k].split(".")[0] == "down":
        bar_speed = -1
        df["phase"] = 0
    else:
        bar_speed_str = stems[k].split(".")[0][3:]
        bar_speed = int(bar_speed_str) / (10 ** (len(bar_speed_str) - 1))
        period_time = 100 / bar_speed

    df["phase"] = (
        (df["time"] - (df["um_y"] / bar_speed)) % (period_time)
    ) / period_time

    df_cropped = df[
        df["time"].between(180, 900)
        & df["px_x"].between(150, 650)
        & df["px_y"].between(150, 650)
        & (df["dtot"] < 2)
        & (~df["dx"].isna())
        & (~df["dy"].isna())
    ].copy()

    print(len(df_cropped))

    df_cropped["phase_bin"] = (df_cropped["phase"] // 0.05) * 0.05

    # sns.lineplot(df_cropped, x="phase_bin", y="dy", label=bar_speed)

    data["vertical speed mean"].append(df_cropped["dy"].mean())
    data["bar speed"].append(bar_speed)

speed_data = pd.DataFrame(data)
sns.lineplot(speed_data, x="bar speed", y="vertical speed mean")
plt.show()

In [ ]:
import cv2
import tifffile
from scipy.spatial.distance import cdist

k = 2

n_samples = 30
time = 300
time_limit = 60
sigma = 25

df = tracks[stems[k]]

df["theta"] = np.arctan2(df["um_x_centered"], df["um_y_centered"])
df["track_initial_theta"] = df["track_id"].map(df.groupby("track_id")["theta"].first())
df["track_initial_theta_vec_x"] = np.cos(df["track_initial_theta"])
df["track_initial_theta_vec_y"] = np.sin(df["track_initial_theta"])
df["track_start_time"] = df.groupby("track_id")["time"].transform("min")

df["dx"] = df.groupby("track_id")["um_x"].diff() / frame_min
df["dy"] = df.groupby("track_id")["um_y"].diff() / frame_min
df["dtot"] = np.sqrt(df["dx"] ** 2 + df["dy"] ** 2)

# get data points near time, filter out bad tracks
df = df[(df["time"] - time).abs() < time_limit]
df = df[~df["dx"].isna()]
df = df[~df["dy"].isna()]
df = df[df["dtot"] < 2].copy()
df = df[df["track_start_time"] < time - 200].copy()

# get sampling positions in image coordinates and um coordinates
ax = np.linspace(-img_shape[1] / 2, img_shape[1] / 2, n_samples) * px_um
ay = np.linspace(-img_shape[0] / 2, img_shape[0] / 2, n_samples) * px_um

px_x = np.linspace(0, img_shape[1], n_samples)
px_y = np.linspace(0, img_shape[0], n_samples)

xx, yy = np.meshgrid(ax, ay)
X = np.array([xx.flatten(), yy.flatten()]).T

px_xx, px_yy = np.meshgrid(px_x, px_y)
X_px = np.array([px_xx.flatten(), px_yy.flatten()])

# get dxdy at sample positions using gaussian kernels
pos = df[["um_x_centered", "um_y_centered"]].values
distances = cdist(X, pos)
weights = np.exp(-0.5 * np.square(distances) / np.square(sigma))

dx_weighted = np.mean(weights * df["dx"].values[None, :], axis=1) / np.mean(
    weights, axis=1
)
dy_weighted = np.mean(weights * df["dy"].values[None, :], axis=1) / np.mean(
    weights, axis=1
)
theta = np.arctan2(dx_weighted, dy_weighted)
area_weighted = np.mean(weights * df["area"].values[None, :], axis=1) / np.mean(
    weights, axis=1
)
initial_theta_x_weighted = np.mean(
    weights * df["track_initial_theta_vec_x"].values[None, :], axis=1
) / np.mean(weights, axis=1)
initial_theta_y_weighted = np.mean(
    weights * df["track_initial_theta_vec_y"].values[None, :], axis=1
) / np.mean(weights, axis=1)
initial_theta_weighted = np.arctan2(initial_theta_y_weighted, initial_theta_x_weighted)

fig, ax = plt.subplots(figsize=(6, 6))

# show light bars
frame_data, frame_pattern = tifffile.imread(base_path / f"{stems[k][:-7]}.tif")[
    int(time // frame_min)
].astype(int)

vmin, vmax = 1000, 7500
frame_data = np.clip((frame_data - vmin) / (vmax - vmin), 0, 1)
# ax.imshow(np.stack([frame_data, frame_data, frame_data], axis=-1))
# ax.imshow(np.stack([np.zeros_like(frame_pattern), frame_pattern, frame_pattern, frame_pattern // 2], axis=-1))

# plot velocities
q = ax.quiver(
    X_px[0],
    X_px[1],
    dx_weighted * 50,
    dy_weighted * 50,
    color="k",
    scale_units="xy",
    clim=(-np.pi, np.pi),
    angles="xy",
    scale=1,
)

ax.axis("equal")
# ax.set_xlim(150, 650)
# ax.set_ylim(150, 650)
ax.set_xticks([])
ax.set_yticks([])
ax.invert_yaxis()


plt.savefig(save_path / f"local_movement_{stems[k]}.pdf", dpi=300)
plt.show()


# # get data points near time, filter out bad tracks
# df = df[(df["time"] - time).abs() < time_limit]
# df = df[~df["dx"].isna()]
# df = df[~df["dy"].isna()]
# df = df[df["dtot"] < 2].copy()
# # df = df[df["track_start_time"] < time - 200].copy()
#
# # get sampling positions in image coordinates and um coordinates
# ax = np.linspace(-img_shape[1] / 2, img_shape[1] / 2, n_samples) * px_um
# ay = np.linspace(-img_shape[0] / 2, img_shape[0] / 2, n_samples) * px_um
#
# px_x = np.linspace(0, img_shape[1], n_samples)
# px_y = np.linspace(0, img_shape[0], n_samples)
#
# xx, yy = np.meshgrid(ax, ay)
# X = np.array([xx.flatten(), yy.flatten()]).T
#
# px_xx, px_yy = np.meshgrid(px_x, px_y)
# X_px = np.array([px_xx.flatten(), px_yy.flatten()])
#
# # get dxdy at sample positions using gaussian kernels
# pos = df[["um_x_centered", "um_y_centered"]].values
# distances = cdist(X, pos)
# weights = np.exp(-0.5 * np.square(distances) / np.square(sigma))
#
#
# dx_weighted = np.mean(weights * df["dx"].values[None, :], axis=1) / np.mean(weights, axis=1)
# dy_weighted = np.mean(weights * df["dy"].values[None, :], axis=1) / np.mean(weights, axis=1)
# theta = np.arctan2(dx_weighted, dy_weighted)
# area_weighted = np.mean(weights * df["area"].values[None, :], axis=1) / np.mean(weights, axis=1)
# initial_theta_x_weighted = np.mean(weights * df["track_initial_theta_vec_x"].values[None, :], axis=1) / np.mean(weights, axis=1)
# initial_theta_y_weighted = np.mean(weights * df["track_initial_theta_vec_y"].values[None, :], axis=1) / np.mean(weights, axis=1)
# initial_theta_weighted = np.arctan2(initial_theta_y_weighted, initial_theta_x_weighted)

import colorstamps

rgb, stamp = colorstamps.apply_stamp(
    dx_weighted.reshape(n_samples, n_samples) * 60,
    dy_weighted.reshape(n_samples, n_samples) * 60,
    "peak",
    vmin_0=-20,
    vmax_0=20,
    vmin_1=-20,
    vmax_1=20,
)
fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)
ax.imshow(rgb, extent=(0, 800, 0, 800))
# q = ax.quiver(X_px[0], X_px[1], dx_weighted*20, dy_weighted*20, color="k", scale_units="xy",angles="xy", scale=1)

ax.set_xlim(150, 650)
ax.set_ylim(150, 650)
ax.set_xticks([])
ax.set_yticks([])
ax.invert_yaxis()

plt.savefig(save_path / f"local_movement_colorquiver_{stems[k]}.pdf", dpi=300)
plt.show()

# also show colormap as in separate ax to illustrate functionality
fig, ax = plt.subplots(1, 1, figsize=(4, 4), dpi=300)

stamp.show_in_ax(ax)
ax.set_ylabel(r"$dy$")
ax.set_xlabel(r"$dx$")
ax.axis("off")
plt.savefig(save_path / f"local_movement_colorquiver_colormap_{stems[k]}.pdf", dpi=300)
plt.show()